# Swedish article choice after a two-parameter edit
Run all cells on CPU. The fixed checkpoint is downloaded the first time. See the research note for the observed results.

In [ ]:
%pip -q install torch==2.14.1 transformers==5.22.1

In [ ]:
"""Equalize the en/ett decoder biases. Run on a free CPU Colab runtime."""
import csv
import json
import os
import platform
from collections import defaultdict
from pathlib import Path

import torch
import transformers
from transformers import AutoModelForMaskedLM, AutoTokenizer

MODEL = "KBLab/bert-base-swedish-cased"
REV = os.environ.get("MODEL_REVISION", "ce7c3424687f042f1320e0528293d492c82918c4")
torch.manual_seed(17)
torch.set_num_threads(min(4, torch.get_num_threads()))
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REV)
model = AutoModelForMaskedLM.from_pretrained(MODEL, revision=REV).cpu().eval()
bias = model.get_output_embeddings().bias
assert bias is not None
original = bias.detach().clone()

def token_id(word):
    ids = tokenizer.encode(word, add_special_tokens=False)
    assert len(ids) == 1 and ids[0] != tokenizer.unk_token_id, word
    return ids[0]

en, ett = map(token_id, ("en", "ett"))
nouns = {
    "dev": {"en": "stol bok bil säng dörr penna", "ett": "hus bord rum brev skåp ägg"},
    "test": {"en": "katt hund nyckel cykel båt sko jacka lampa väska spegel kopp klocka fågel blomma tidning flaska",
             "ett": "fönster glas tåg barn träd skepp ben fat rep tak golv ljus moln lås hjul mynt"},
}
frames = ["Jag såg {} {}.", "Här finns {} {}."]
regression = [
    ("Hon köpte en bok och började [MASK].", "läsa"),
    ("På vintern kan det falla [MASK].", "snö"),
    ("Han drack ett glas [MASK].", "vatten"),
    ("Solen går upp i [MASK].", "öster"),
    ("Klockan är fem på [MASK].", "morgonen"),
    ("De gick till sängs för att [MASK].", "sova"),
    ("Han tog på sig sina [MASK].", "skor"),
    ("Sverige ligger i [MASK].", "Europa"),
    ("De väntade i [MASK] timme.", "en"),
    ("Hon skrev [MASK] brev till mig.", "ett"),
    ("Vi såg [MASK] film igår.", "en"),
    ("De öppnade [MASK] fönster.", "ett"),
]
examples = [(split, noun, frame.format("[MASK]", noun), article)
            for split, groups in nouns.items() for article, words in groups.items()
            for noun in words.split() for frame in frames]
examples += [("regression", str(i), prompt, gold) for i, (prompt, gold) in enumerate(regression)]
assert set(r[1] for r in examples if r[0]=="dev").isdisjoint(r[1] for r in examples if r[0]=="test")

@torch.inference_mode()
def score(records):
    chunks = []
    for start in range(0, len(records), 12):
        texts = [r[2] for r in records[start:start+12]]
        inputs = tokenizer(texts, padding=True, return_tensors="pt")
        mask = inputs.input_ids == tokenizer.mask_token_id
        assert (mask.sum(1) == 1).all()
        chunks.append(model(**inputs).logits[mask])
    return torch.cat(chunks)

# Only development examples enter threshold selection. No test feedback.
dev = [r for r in examples if r[0] == "dev"]
held = [r for r in examples if r[0] != "dev"]
dev_logits = score(dev)
margins = dev_logits[:, en] - dev_logits[:, ett]
grid = [i / 4 for i in range(-24, 25)]
def dev_hits(offset):
    return sum(bool(m + offset >= 0) == (r[3] == "en") for m, r in zip(margins, dev))
fitted = max(grid, key=lambda v: (dev_hits(v), -abs(v)))
gap = float(original[en] - original[ett])
rows = []
metrics = defaultdict(list)
for condition, offset in [("baseline", 0), ("equalized", -gap), ("opposite", gap), ("dev_calibrated", fitted)]:
    with torch.no_grad():
        bias.copy_(original)
        # Equal and opposite shifts preserve the pair's mean logit.
        bias[en] += offset / 2
        bias[ett] -= offset / 2
    logits = score(held)
    for (split, item, prompt, gold), vec in zip(held, logits):
        margin = float(vec[en] - vec[ett])
        pred = ("en" if margin >= 0 else "ett") if split == "test" else tokenizer.convert_ids_to_tokens(int(vec.argmax()))
        gold_id = token_id(gold)
        logp = float(vec.log_softmax(0)[gold_id])
        rows.append(dict(condition=condition, split=split, item=item, text=prompt,
                         gold=gold, prediction=pred, correct=int(pred==gold), margin=margin, gold_logp=logp))
        metrics[(condition, split)].append((int(pred==gold), logp))
with torch.no_grad():
    bias.copy_(original)
out = Path("results")
out.mkdir(exist_ok=True)
with (out / "predictions.csv").open("w", newline="", encoding="utf8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
config = dict(model=MODEL, revision=REV, torch=torch.__version__, transformers=transformers.__version__,
              python=platform.python_version(), seed=17, device="cpu", bias_gap=gap, dev_offset=fitted)
print(json.dumps(config))
(out / "config.json").write_text(json.dumps(config, indent=2))
for key, values in metrics.items():
    print(*key, sum(x for x,y in values), "/", len(values), "mean gold logp", round(sum(y for x,y in values)/len(values), 5))
